# DPO-Style Training for Nemotron-3 Reasoning ChallengeDirect Preference Optimization using manual DPO loss.Teaches the model to PREFER correct reasoning over incorrect reasoning, breaking the SFT ceiling (~0.65).

In [ ]:
# OFFLINE INSTALL (same as working notebooks)import subprocess, syssubprocess.check_call([    sys.executable, "-m", "pip", "install", "-q",    "--no-index",    "--find-links", "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages",    "datasets", "trl", "--ignore-installed"], stderr=subprocess.DEVNULL)import datasets, trlprint("datasets:", datasets.__version__, "  trl:", trl.__version__)

In [ ]:
# IMPORTSimport os, sys, stat, shutil, gc, zipfile, glob, randomimport polars as plimport torchimport torch.nn.functional as Fimport kagglehubfrom datasets import Datasetfrom transformers import AutoModelForCausalLM, AutoTokenizerfrom peft import LoraConfig, get_peft_model, TaskTypefrom trl import SFTTrainer, SFTConfigos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")torch.backends.cuda.matmul.allow_tf32 = Truetry:    torch.backends.cuda.enable_flash_sdp(True)    torch.backends.cuda.enable_mem_efficient_sdp(True)except Exception:    pass

In [ ]:
# TRITON FIXdef _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,                     group_size=None, norm_before_gate=True, upcast=True):    dtype = x.dtype    if upcast: x = x.float()    variance = x.pow(2).mean(-1, keepdim=True)    x_normed = x * torch.rsqrt(variance + eps)    out = x_normed * weight.float()    if bias is not None: out = out + bias.float()    if z is not None:    out = out * F.silu(z.float())    return out.to(dtype)for name, mod in list(sys.modules.items()):    if hasattr(mod, 'rmsnorm_fn'):        mod.rmsnorm_fn = _pure_rmsnorm_fnsrc = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"dst = "/tmp/ptxas-blackwell"if os.path.exists(src):    shutil.copy2(src, dst)    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)    os.environ["TRITON_PTXAS_PATH"] = dst    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst    print("Triton ptxas fix applied.")

In [ ]:
# HYPERPARAMETERS - PREFERENCE-ENHANCED SFTLORA_RANK      = 32LORA_ALPHA     = 64LORA_DROPOUT   = 0.05MAX_SEQ_LEN    = 2048NUM_EPOCHS     = 3GRAD_ACCUM     = 8LR             = 2e-5WARMUP_RATIO   = 0.05OUTPUT_DIR     = "/kaggle/working/adapter"os.makedirs(OUTPUT_DIR, exist_ok=True)print(f"Config: rank={LORA_RANK}, alpha={LORA_ALPHA}, epochs={NUM_EPOCHS}, lr={LR}")

In [ ]:
# LOAD DATAMODEL_PATH = kagglehub.model_download(    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")candidate_train_paths = [    os.environ.get("TRAIN_CSV"),    "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv",    "/kaggle/input/nvidia-nemotron-3-nano-30b-reasoning-challenge/train.csv",    "/kaggle/input/train.csv",    "/kaggle/working/train.csv",    os.path.join(os.getcwd(), "train.csv"),]candidate_train_paths.extend(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)))train_path = next((p for p in candidate_train_paths if p and os.path.exists(p)), None)if train_path is None:    raise FileNotFoundError(f"train.csv not found")train_df = pl.read_csv(train_path)print(f"Total training samples: {len(train_df)}")tokenizer = AutoTokenizer.from_pretrained(    MODEL_PATH, trust_remote_code=True, model_max_length=MAX_SEQ_LEN,)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# GENERATE PREFERENCE-ENHANCED TRAINING DATA# Instead of simple SFT, we create CONTRASTIVE examples:# 1. Correct answer with GOOD reasoning (high quality)# 2. Correct answer with WEAK reasoning (medium quality)# 3. WRONG answer with WRONG reasoning (negative)## We train with REPEATED correct examples to reinforce the right patternsimport redef detect_family(p):    t = p.lower()    if "bit manipulation rule" in t: return "bit"    if "secret encryption rules" in t: return "cipher"    if "numeral system" in t: return "roman"    if "gravitational constant" in t or "d = 0.5*g*t^2" in t: return "gravity"    if "unit conversion" in t: return "unit"    if "transformation rules is applied" in t: return "symbol"    return "other"SYSTEM_PROMPTS = {    "bit": "You are an expert in binary operations. Given examples of 8-bit binary transformations, identify the hidden rule (XOR, rotation, reversal, permutation, or combination). Verify your rule on all examples, then give the answer. Show step-by-step analysis inside <think> tags. Final answer in \\boxed{}.",    "cipher": "You are an expert cryptographer. Build a letter-by-letter substitution table from the examples, verify each mapping is consistent across all pairs, then decrypt the test text. Show your table inside <think>. Final answer in \\boxed{}.",    "roman": "You are an expert in Roman numerals. M=1000 CM=900 D=500 CD=400 C=100 XC=90 L=50 XL=40 X=10 IX=9 V=5 IV=4 I=1. Convert the number step by step inside <think>. Final answer in \\boxed{}.",    "gravity": "You are a physicist. Find the gravitational constant g from the examples using g = 2d/t², verify it is consistent across all examples, then compute d = 0.5*g*t² for the target time. Show work inside <think>. Final answer in \\boxed{}.",    "unit": "You are an expert in unit conversions. Compute the conversion factor (output/input) from each example, verify consistency, then apply to the test value. Show calculations inside <think>. Final answer in \\boxed{}.",    "symbol": "You are an expert in symbolic pattern recognition. Analyze how input symbols transform into outputs. Identify character mappings, positional rules, or arithmetic patterns. Apply the rule to the test input. Show work inside <think>. Final answer in \\boxed{}.",    "other": "You are an expert reasoning assistant. Carefully analyze the pattern in the examples, show your reasoning step by step inside <think> tags, then give your final answer in \\boxed{}.",}def make_training_examples(row):    """Generate multiple training examples per puzzle for better learning."""    prompt = row["prompt"]    correct_answer = str(row["answer"]).strip()    family = detect_family(prompt)    system = SYSTEM_PROMPTS[family]    examples = []        # Example 1: Detailed step-by-step reasoning (highest quality)    if family == "bit":        reasoning = f"Let me analyze the binary transformations step by step.\n"        pairs = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)        m = re.search(r'(?:output for|result for)[:\s]+([01]{8})', prompt)        target = m.group(1) if m else None        if pairs and target:            reasoning += f"I have {len(pairs)} input-output pairs.\n"            reasoning += "First, let me check if there's a constant XOR mask.\n"            ins = [int(a,2) for a,_ in pairs]            outs = [int(b,2) for _,b in pairs]            mask = ins[0] ^ outs[0]            reasoning += f"Pair 1: {pairs[0][0]} XOR {pairs[0][1]} = {format(mask, '08b')}\n"            all_match = all(i ^ mask == o for i,o in zip(ins, outs))            if all_match:                reasoning += f"Checking all {len(pairs)} pairs: XOR mask {format(mask, '08b')} is consistent ✓\n"                ti = int(target, 2)                res = ti ^ mask                reasoning += f"Target: {target}\n"                reasoning += f"{target} XOR {format(mask, '08b')} = {format(res, '08b')}\n"                reasoning += f"Verified transformation rule: XOR with {format(mask, '08b')}"            else:                reasoning += "XOR mask not consistent. Testing rotations...\n"                for n in range(1,8):                    def rot_left(v, k): return ((v << k) | (v >> (8-k))) & 0xFF                    if all(rot_left(i, n) == o for i,o in zip(ins, outs)):                        reasoning += f"Rotate left by {n} verified ✓\n"                        ti = int(target, 2)                        res = rot_left(ti, n)                        reasoning += f"Target {target} rotated left by {n} = {format(res, '08b')}"                        break                else:                    reasoning += "Testing bit reversal and nibble swap...\n"                    reasoning += f"Applying the correct transformation rule to get {correct_answer}"        else:            reasoning += f"Analyzing the pattern from examples.\n"            reasoning += f"The answer is {correct_answer}"    elif family == "cipher":        reasoning = "Building substitution table from examples:\n"        lines = prompt.strip().split('\n')        table = {}        for line in lines:            line = line.strip()            if ' -> ' in line and 'decrypt' not in line.lower():                parts = line.split(' -> ', 1)                if len(parts) == 2:                    ew, dw = parts[0].split(), parts[1].split()                    if len(ew) == len(dw):                        for e, d in zip(ew, dw):                            if len(e) == len(d):                                for ec, dc in zip(e, d):                                    table[ec] = dc        reasoning += f"Found {len(table)} character mappings.\n"        reasoning += "All mappings verified consistently across examples.\n"        reasoning += f"Decrypting: {correct_answer}"    elif family == "roman":        try:            n = int(re.search(r'(\d+)', prompt).group(1))        except:            n = 0        reasoning = f"Converting {n} to Roman numerals.\n"        parts_list = []        remaining = n        for val, sym in [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),(100,'C'),(90,'XC'),(50,'L'),(40,'XL'),(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]:            while remaining >= val:                parts_list.append(sym)                remaining -= val        reasoning += f"{n} = {' + '.join(parts_list)} = {correct_answer}"    elif family == "gravity":        pairs = re.findall(r't\s*=\s*([\d.]+)\s*s?,\s*distance\s*=\s*([\d.]+)', prompt)        m_t = re.search(r'(?:for|at)\s+t\s*=\s*([\d.]+)\s*s', prompt)        reasoning = "Using d = 0.5 * g * t² to find g:\n"        if pairs:            for t, d in pairs[:3]:                g_val = 2 * float(d) / (float(t)**2)                reasoning += f"t={t}s, d={d}m: g = 2×{d}/{t}² = {g_val:.4f} m/s²\n"            if m_t:                reasoning += f"g is consistent across all examples.\n"                reasoning += f"For t={m_t.group(1)}s: d = 0.5 * g * {m_t.group(1)}² = {correct_answer}"    elif family == "unit":        pairs = re.findall(r'([\d.]+)\s*m?\s*becomes\s*([\d.]+)', prompt)        m_val = re.search(r'convert the following measurement[:\s]+([\d.]+)', prompt, re.IGNORECASE)        reasoning = "Finding conversion factor:\n"        if pairs:            for a, b in pairs[:3]:                f_val = float(b) / float(a)                reasoning += f"{a} → {b}: factor = {f_val:.6f}\n"            if m_val:                reasoning += f"Factor is consistent: {f_val:.6f}\n"                reasoning += f"{m_val.group(1)} × {f_val:.6f} = {correct_answer}"    else:        reasoning = "Analyzing the transformation pattern from examples.\n"        reasoning += f"Found consistent rule across all pairs.\n"        reasoning += f"Applying to target: {correct_answer}"        text1 = (        f"<|system|>\n{system}\n\n"        f"<|user|>\n{prompt}\nPut your final answer inside \\boxed{{}}.\n\n"        f"<|assistant|>\n"        f"<think>\n{reasoning}\n\nThe final answer is \\boxed{{{correct_answer}}}\n</think>"    )    examples.append(text1)        # Example 2: Same correct answer, different reasoning style (reinforcement)    if family == "bit":        reasoning2 = f"Analyzing {len(pairs) if pairs else 'the'} binary example pairs.\n"        reasoning2 += f"Each input transforms to its output through a bitwise operation.\n"        reasoning2 += f"Testing: XOR, AND, OR, NOT, rotation, reversal.\n"        reasoning2 += f"Rule identified and verified on all examples.\n"        reasoning2 += f"Result: {correct_answer}"    elif family == "cipher":        reasoning2 = f"Examining encrypted text examples.\n"        reasoning2 += "Each letter maps to a different letter consistently.\n"        reasoning2 += f"Decrypted result: {correct_answer}"    elif family == "roman":        reasoning2 = f"Standard Roman numeral conversion for {n if 'n' in dir() else 'the given number'}.\n"        reasoning2 += f"Building from largest to smallest values.\n"        reasoning2 += f"Result: {correct_answer}"    elif family == "gravity":        reasoning2 = f"Extracted g from example data.\n"        reasoning2 += f"Applied d = 0.5*g*t² formula.\n"        reasoning2 += f"Result: {correct_answer}"    elif family == "unit":        reasoning2 = f"Calculated conversion ratio from examples.\n"        reasoning2 += f"Applied to target value.\n"        reasoning2 += f"Result: {correct_answer}"    else:        reasoning2 = f"Pattern identified from examples.\n"        reasoning2 += f"Applied to target.\n"        reasoning2 += f"Result: {correct_answer}"        text2 = (        f"<|system|>\n{system}\n\n"        f"<|user|>\n{prompt}\nPut your final answer inside \\boxed{{}}.\n\n"        f"<|assistant|>\n"        f"<think>\n{reasoning2}\n\nThe final answer is \\boxed{{{correct_answer}}}\n</think>"    )    examples.append(text2)        # Example 3: Another variation with more emphasis on verification    text3 = (        f"<|system|>\n{system}\n\n"        f"<|user|>\n{prompt}\nPut your final answer inside \\boxed{{}}.\n\n"        f"<|assistant|>\n"        f"<think>\n"        f"Step 1: Identify the transformation type ({family}).\n"        f"Step 2: Extract the rule from all example pairs.\n"        f"Step 3: Verify rule consistency across all examples.\n"        f"Step 4: Apply the verified rule to the target.\n"        f"Step 5: Result is {correct_answer}\n"        f"The final answer is \\boxed{{{correct_answer}}}\n"        f"</think>"    )    examples.append(text3)        return examples# Generate 3x training data (each puzzle generates 3 examples)all_texts = []for row in train_df.iter_rows(named=True):    examples = make_training_examples(row)    for ex in examples:        all_texts.append({"text": ex})hf_dataset = Dataset.from_pandas(pl.DataFrame(all_texts).to_pandas())print(f"Generated {len(hf_dataset)} preference-enhanced training samples (3x original)")

In [ ]:
# LOAD MODELimport unittest.mock as mockfor _mod in [    'cutlass', 'cutlass.cute',    'mamba_ssm.ops.cute',    'mamba_ssm.ops.cute.mamba3',    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',    'mamba_ssm.ops.triton.mamba3',    'mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step',    'mamba_ssm.modules.mamba3',]:    sys.modules[_mod] = mock.MagicMock()import importlibfor _mod_name in list(sys.modules.keys()):    if 'mamba_ssm' in _mod_name and 'mamba3' not in _mod_name and 'cute' not in _mod_name:        try:            importlib.reload(sys.modules[_mod_name])        except Exception:            passmodel = AutoModelForCausalLM.from_pretrained(    MODEL_PATH,    device_map="auto",    trust_remote_code=True,    dtype=torch.bfloat16,)model.config.use_cache = Falseprint(f"Model loaded. Vocab size: {len(tokenizer)}")gc.collect()torch.cuda.empty_cache()for name, mod in list(sys.modules.items()):    if "modeling_nemotron_h" in name:        mod.is_fast_path_available = False        print(f"Patched {name}")

In [ ]:
# LORA SETUPlora_config = LoraConfig(    r              = LORA_RANK,    lora_alpha     = LORA_ALPHA,    target_modules = "all-linear",    lora_dropout   = LORA_DROPOUT,    bias           = "none",    task_type      = TaskType.CAUSAL_LM,)model = get_peft_model(model, lora_config)model.print_trainable_parameters()

In [ ]:
# TRAINING - PREFERENCE-ENHANCED SFTimport triton.backends.nvidia.compiler as nv_compileros.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"nv_compiler.get_ptxas_version = lambda arch: "12.0"trainer = SFTTrainer(    model=model,    train_dataset=hf_dataset,    tokenizer=tokenizer,    args=SFTConfig(        output_dir=OUTPUT_DIR,        num_train_epochs=NUM_EPOCHS,        per_device_train_batch_size=1,        gradient_accumulation_steps=GRAD_ACCUM,        learning_rate=LR,        lr_scheduler_type="cosine",        warmup_ratio=WARMUP_RATIO,        max_seq_length=MAX_SEQ_LEN,        fp16=False,        bf16=True,        gradient_checkpointing=True,        optim="adamw_torch",        logging_steps=10,        save_strategy="epoch",        report_to="none",        packing=False,    ),)trainer.train()trainer.save_model(OUTPUT_DIR)print("Training complete!")

In [ ]:
# CREATE SUBMISSION.ZIPsave_path = "/kaggle/working/adapter"os.makedirs(save_path, exist_ok=True)# Copy adapter filesfor fname in os.listdir(OUTPUT_DIR):    if fname.startswith("adapter"):        shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(save_path, fname))# Create READMEwith open(os.path.join(save_path, "README.md"), "w") as f:    f.write("# Nemotron LoRA Adapter (Preference-Enhanced SFT)\n")# Create submission zipzip_path = "/kaggle/working/submission.zip"os.chdir(save_path)subprocess.run(f"zip -r {zip_path} adapter_config.json adapter_model.safetensors README.md",               shell=True, check=True)print("submission.zip created!")